# 01_eda — H&M Personalized Fashion Recommendations

`01_input/` に配置済みのデータに対する基本EDA(`head`, `info`, `describe`, `isnull`, `nunique`)。

## データ概要
- articles.csv: 25列(商品マスタ)
- customers.csv: 7列(顧客マスタ)
- transactions_train.csv: 5列(購買履歴、約3,100万行)
- sample_submission.csv: 2列(提出フォーマット)
- images/: 商品画像 約105,100枚(.jpg)


In [ ]:
import os
import glob
import pandas as pd

DATA_DIR = "../01_input"

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
jpg_files = glob.glob(os.path.join(DATA_DIR, "**", "*.jpg"), recursive=True)

print("CSVファイル:")
for f in csv_files:
    print(" -", f)
print(f"\n画像ファイル数(.jpg): {len(jpg_files):,}")


---
## articles.csv(商品マスタ)

In [ ]:
articles = pd.read_csv(os.path.join(DATA_DIR, "articles.csv"))
articles.head()


In [ ]:
articles.info()

In [ ]:
articles.describe(include="all").T

In [ ]:
articles.isnull().sum().sort_values(ascending=False)

In [ ]:
articles.nunique().sort_values(ascending=False)

---
## customers.csv(顧客マスタ)

In [ ]:
customers = pd.read_csv(os.path.join(DATA_DIR, "customers.csv"))
customers.head()


In [ ]:
customers.info()

In [ ]:
customers.describe(include="all").T

In [ ]:
customers.isnull().sum().sort_values(ascending=False)

In [ ]:
customers.nunique().sort_values(ascending=False)

---
## transactions_train.csv(購買履歴)

約3,100万行あるため読み込みに時間・メモリがかかります。必要なら `nrows=` で件数を絞ってください。

In [ ]:
# 全件読み込み(メモリに余裕がない場合は nrows=1_000_000 などに変更)
transactions = pd.read_csv(os.path.join(DATA_DIR, "transactions_train.csv"))
transactions.head()


In [ ]:
transactions.info()

In [ ]:
transactions.describe(include="all").T

In [ ]:
transactions.isnull().sum().sort_values(ascending=False)

In [ ]:
transactions.nunique().sort_values(ascending=False)

---
## sample_submission.csv(提出フォーマット)

In [ ]:
sample_submission = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
sample_submission.head()


In [ ]:
sample_submission.info()

In [ ]:
sample_submission.isnull().sum()

In [ ]:
sample_submission.nunique()

---
## 画像データの確認(サンプル表示)

画像は `images/0{department_no}/{article_id}.jpg` の形式(約1166×1750ピクセル)。全件読み込みは重いのでサンプルのみ表示。

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

sample_paths = jpg_files[:6]

if sample_paths:
    fig, axes = plt.subplots(1, len(sample_paths), figsize=(18, 4))
    for ax, path in zip(axes, sample_paths):
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(os.path.basename(path), fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("images フォルダが見つかりません。ダウンロード・解凍を確認してください。")


---
## 基本EDAからの次のステップ

- テーブル結合キー: `transactions_train.article_id` ↔ `articles.article_id` / `transactions_train.customer_id` ↔ `customers.customer_id`
- 購買頻度分布・リピート購入率・季節性は、以降の「仮説駆動EDA(H1〜H7 可視化)」セクションで検証済み
- 結果や気づきは `experiment_log.csv` に記録

# 仮説駆動EDA(H1〜H7 可視化)

`00_md/hypothesis_driven_design.md` で duckdb 集計により数値検証済みの仮説(H1〜H7)を、
このノートブック上で再現し可視化する。H6(送料無料閾値)はprice列が匿名化スケーリング値のため直接検証不可のため対象外。

3,100万行のtransactionsを素朴にpandasでgroup byするとメモリ制約(約3.8GB)でOOMするため、
`customer_id`(64文字文字列)を整数サロゲートキー化した上でduckdbで集計する(詳細はメモ8節)。
まず`02_output/features/trans_lite.parquet`を作成し、以降はこれに対してduckdbでSQLを投げる。

In [ ]:
import duckdb

TRANS_LITE_PATH = os.path.join("..", "02_output", "features", "trans_lite.parquet")

con = duckdb.connect()
con.execute("PRAGMA threads=4")

if not os.path.exists(TRANS_LITE_PATH):
    con.execute(f"""
    CREATE TABLE cust_map AS
    SELECT customer_id, (ROW_NUMBER() OVER () - 1)::INTEGER AS customer_idx
    FROM read_csv_auto('{DATA_DIR}/customers.csv')
    """)
    con.execute(f"""
    COPY (
      SELECT cm.customer_idx, t.t_dat,
             CAST(t.article_id AS BIGINT) AS article_id,
             t.price, t.sales_channel_id::TINYINT AS sales_channel_id,
             CAST(a.product_code AS INTEGER) AS product_code,
             a.product_group_name, a.index_group_name,
             c.age::SMALLINT AS age
      FROM read_csv_auto('{DATA_DIR}/transactions_train.csv') t
      JOIN cust_map cm ON cm.customer_id = t.customer_id
      LEFT JOIN read_csv_auto('{DATA_DIR}/articles.csv') a
        ON CAST(a.article_id AS BIGINT) = CAST(t.article_id AS BIGINT)
      LEFT JOIN read_csv_auto('{DATA_DIR}/customers.csv') c
        ON c.customer_id = t.customer_id
    ) TO '{TRANS_LITE_PATH}' (FORMAT PARQUET)
    """)
    print("trans_lite.parquet を作成しました")
else:
    print("trans_lite.parquet は作成済みのため読み込みのみ行います")

con.execute(f"CREATE OR REPLACE VIEW trans AS SELECT * FROM read_parquet('{TRANS_LITE_PATH}')")
con.execute("SELECT COUNT(*) FROM trans").df()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

# 配色(カテゴリカル固定順)。店舗=BLUE, オンライン=ORANGE で全チャート共通にする
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = \
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"
PRIMARY, SECONDARY, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"

plt.rcParams.update({
    "font.family": "Hiragino Sans",
    "axes.edgecolor": GRID,
    "axes.labelcolor": SECONDARY,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "text.color": PRIMARY,
    "axes.grid": True,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "axes.axisbelow": True,
})

def style_ax(ax, hide_y_grid=False):
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    if hide_y_grid:
        ax.yaxis.grid(False)


---
## H1: 同じ商品/色違いを買う傾向がある(採用)

各購入イベントについて「そのarticle_idを過去に同顧客が買ったことがあるか」「同一product_code(色違い)を過去に買ったことがあるか」を判定し、3区分の割合を集計する。

In [ ]:
h1 = con.execute("""
WITH cat AS (
  SELECT
    CASE
      WHEN ROW_NUMBER() OVER (PARTITION BY customer_idx, article_id ORDER BY t_dat) > 1 THEN '完全同一article再購入'
      WHEN ROW_NUMBER() OVER (PARTITION BY customer_idx, product_code ORDER BY t_dat) > 1 THEN '同一product_code色違い'
      ELSE '新規商品'
    END AS category
  FROM trans
)
SELECT category, COUNT(*) AS n, COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS pct
FROM cat GROUP BY category ORDER BY n DESC
""").df()
h1 = h1.set_index("category").loc[["新規商品", "完全同一article再購入", "同一product_code色違い"]].reset_index()
print(h1)

fig, ax = plt.subplots(figsize=(7, 3))
colors = [BLUE, ORANGE, AQUA]
bars = ax.barh(h1["category"], h1["pct"], color=colors, height=0.55)
ax.invert_yaxis()
style_ax(ax, hide_y_grid=True)
ax.set_xlim(0, h1["pct"].max() * 1.15)
ax.set_xlabel("購入イベントに占める割合 (%)")
for b, v in zip(bars, h1["pct"]):
    ax.text(v + h1["pct"].max() * 0.02, b.get_y() + b.get_height() / 2, f"{v:.1f}%",
            va="center", color=SECONDARY, fontsize=10)
ax.set_title("H1: 購入商品の新規性 vs リピート傾向", loc="left", fontsize=12, color=PRIMARY)
fig.tight_layout()
plt.show()


---
## H2: カテゴリ別リピート嗜好(Menswear/Baby-Children採用・強い / Ladieswear棄却・弱い)

各index_groupを3回以上購入した顧客について、「その顧客の全購入に占める当該index_groupの割合」を顧客ごとに計算し平均する(個人内シェアの平均)。母集団平均シェアと比較することで、単なる母集団の多数派か、真に判別力のある嗜好かを区別する。

In [ ]:
pop_share = con.execute("""
SELECT index_group_name, COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS pop_pct
FROM trans GROUP BY index_group_name
""").df().set_index("index_group_name")["pop_pct"]

def heavy_personal_share(group_name, min_count=3):
    q = f"""
    WITH per_cust AS (
      SELECT customer_idx,
             COUNT(*) FILTER (WHERE index_group_name = '{group_name}') AS grp_cnt,
             COUNT(*) AS total_cnt
      FROM trans GROUP BY customer_idx
    )
    SELECT COUNT(*) AS n_heavy, AVG(grp_cnt * 1.0 / total_cnt) * 100 AS avg_personal_share_pct
    FROM per_cust WHERE grp_cnt >= {min_count}
    """
    return con.execute(q).df().iloc[0]

groups = ["Menswear", "Baby/Children", "Ladieswear"]
heavy_pct = [heavy_personal_share(g)["avg_personal_share_pct"] for g in groups]
pop_pct = [pop_share[g] for g in groups]

x = np.arange(len(groups))
w = 0.35
fig, ax = plt.subplots(figsize=(7.5, 4))
b1 = ax.bar(x - w / 2, pop_pct, width=w, color=BLUE, label="母集団平均シェア")
b2 = ax.bar(x + w / 2, heavy_pct, width=w, color=ORANGE, label="当該カテゴリ3回以上購入者の個人内平均シェア")
ax.set_xticks(x)
ax.set_xticklabels(groups)
style_ax(ax, hide_y_grid=False)
ax.xaxis.grid(False)
ax.set_ylabel("購入点数に占めるシェア (%)")
for bars in (b1, b2):
    for b in bars:
        h = b.get_height()
        ax.text(b.get_x() + b.get_width() / 2, h + 1, f"{h:.1f}%", ha="center", fontsize=9, color=SECONDARY)
ax.legend(frameon=False, loc="upper left", fontsize=9)
ax.set_title("H2: カテゴリ別リピート嗜好の強さ(母集団比)", loc="left", fontsize=12, color=PRIMARY)
fig.tight_layout()
plt.show()

print("倍率(個人内シェア / 母集団シェア):", {g: round(h / p, 2) for g, h, p in zip(groups, heavy_pct, pop_pct)})


---
## H3: 店舗ではアクセサリ・靴下など付随購入が起きやすい(採用・効果は中程度)

(customer_idx, t_dat, sales_channel_id)を1バスケットとみなし、衣類(Garment Upper/Lower/Full body)を含むバスケットのうち、付随カテゴリ(Accessories/Socks & Tights/Underwear)を併買しているものの割合をチャネル別に集計する。

In [ ]:
h3 = con.execute("""
WITH basket AS (
  SELECT customer_idx, t_dat, sales_channel_id,
         BOOL_OR(product_group_name IN ('Garment Upper body', 'Garment Lower body', 'Garment Full body')) AS has_clothing,
         BOOL_OR(product_group_name IN ('Accessories', 'Socks & Tights', 'Underwear')) AS has_accessory
  FROM trans
  GROUP BY customer_idx, t_dat, sales_channel_id
)
SELECT sales_channel_id,
       COUNT(*) FILTER (WHERE has_clothing) AS n_clothing_baskets,
       SUM(CASE WHEN has_clothing AND has_accessory THEN 1 ELSE 0 END) * 100.0
         / COUNT(*) FILTER (WHERE has_clothing) AS accessory_attach_pct
FROM basket
GROUP BY sales_channel_id ORDER BY sales_channel_id
""").df()
print(h3)

channels = ["店舗", "オンライン"]
attach = h3.sort_values("sales_channel_id")["accessory_attach_pct"].tolist()

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(channels, attach, color=[BLUE, ORANGE], width=0.5)
style_ax(ax, hide_y_grid=False)
ax.xaxis.grid(False)
ax.set_ylabel("付随購入併買率 (%)")
ax.set_ylim(0, max(attach) * 1.25)
for b, v in zip(bars, attach):
    ax.text(b.get_x() + b.get_width() / 2, v + max(attach) * 0.02, f"{v:.1f}%", ha="center", fontsize=10, color=SECONDARY)
ax.set_title("H3: チャネル別 付随購入率", loc="left", fontsize=12, color=PRIMARY)
fig.tight_layout()
plt.show()


---
## H4: 季節性・トレンドに大きく左右される(採用・強い)

週次の取引件数を集計する。Black Friday週のスパイクと、2020-03-16〜04-27のCOVID-19店舗休業による落ち込みが明確に見えるはず。COVID期は非定常なレジームシフトのため、トレンド系特徴量の学習からは除外 or 別重みを検討する対象。

In [ ]:
h4 = con.execute("""
SELECT date_trunc('week', CAST(t_dat AS DATE)) AS week, COUNT(*) AS n
FROM trans GROUP BY week ORDER BY week
""").df()
h4["week"] = pd.to_datetime(h4["week"])

fig, ax = plt.subplots(figsize=(10, 4.5))
ymax = h4["n"].max() * 1.28
ax.set_ylim(0, ymax)
ax.plot(h4["week"], h4["n"], color=BLUE, linewidth=2)

covid_start, covid_end = pd.Timestamp("2020-03-16"), pd.Timestamp("2020-04-27")
ax.axvspan(covid_start, covid_end, color=RED, alpha=0.08)
ax.text(covid_start, ymax * 0.90, " COVID-19店舗休業期", color=RED, fontsize=9, va="top")

bf = pd.Timestamp("2018-11-19")
bf_y = h4.loc[h4["week"] == bf, "n"].values[0] if (h4["week"] == bf).any() else h4["n"].max()
ax.annotate("Black Friday週", xy=(bf, bf_y),
            xytext=(bf + pd.Timedelta(days=40), ymax * 0.90), fontsize=9, color=SECONDARY,
            arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.8))

style_ax(ax, hide_y_grid=False)
ax.xaxis.grid(False)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.set_ylabel("週次取引件数")
ax.set_title("H4: 週次取引件数の推移(季節性・COVID影響)", loc="left", fontsize=12, color=PRIMARY, pad=12)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()


---
## H5: 自分の予算に合わせた価格帯で買う傾向がある(採用)

顧客ごとの価格CV(std/mean)を計算し、その分布と平均を、全取引を1つのプールとみなした母集団CVと比較する。個人内CVの分布が母集団CVより明確に左(小さい側)に寄っていれば、各顧客が自分の予算レンジ内で買い物をしていることの裏付けになる。

In [ ]:
cv_df = con.execute("""
WITH per_cust AS (
  SELECT customer_idx, AVG(price) AS m, STDDEV_SAMP(price) AS s, COUNT(*) AS n
  FROM trans GROUP BY customer_idx
)
SELECT s / m AS cv FROM per_cust WHERE n >= 2 AND m > 0
""").df()
pop_cv = con.execute("SELECT STDDEV_SAMP(price) / AVG(price) AS pop_cv FROM trans").df().iloc[0]["pop_cv"]
personal_cv_avg = cv_df["cv"].mean()
print(f"顧客個人内CV平均: {personal_cv_avg:.3f} / 母集団CV: {pop_cv:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(cv_df["cv"].clip(upper=1.5), bins=60, color=BLUE, alpha=0.9)
ax.axvline(pop_cv, color=RED, linewidth=2, linestyle="--")
ax.text(pop_cv + 0.02, ax.get_ylim()[1] * 0.92, f"母集団CV = {pop_cv:.3f}", color=RED, fontsize=9, va="top")
ax.axvline(personal_cv_avg, color=ORANGE, linewidth=2, linestyle="--")
ax.text(personal_cv_avg - 0.02, ax.get_ylim()[1] * 0.78, f"顧客個人内CV平均 = {personal_cv_avg:.3f}",
        color=ORANGE, fontsize=9, va="top", ha="right")
style_ax(ax, hide_y_grid=False)
ax.xaxis.grid(False)
ax.set_xlabel("顧客ごとの価格CV(std/mean)")
ax.set_ylabel("顧客数")
ax.set_title("H5: 顧客内価格CV分布 vs 母集団CV", loc="left", fontsize=12, color=PRIMARY)
fig.tight_layout()
plt.show()


---
## H7: 店舗の方が予算が少なく、オンラインの方が高額・多点数を買う(採用)

チャネル別に平均単価・平均バスケット合計額・平均購入点数(バスケットあたり)を比較する。3指標はスケールが異なるため、1つの軸に無理にまとめず(2軸グラフは作らない)、小さい図を3枚並べるsmall multiples形式にする。H6(送料無料閾値)はprice列が匿名化スケーリング値のため直接検証不可だったが、この結果はH6と間接的に整合する。

In [ ]:
h7_price = con.execute("SELECT sales_channel_id, AVG(price) AS avg_price FROM trans GROUP BY 1 ORDER BY 1").df()
h7_basket = con.execute("""
WITH basket AS (
  SELECT customer_idx, t_dat, sales_channel_id, SUM(price) AS basket_total, COUNT(*) AS n_items
  FROM trans GROUP BY 1, 2, 3
)
SELECT sales_channel_id, AVG(basket_total) AS avg_basket_total, AVG(n_items) AS avg_items
FROM basket GROUP BY 1 ORDER BY 1
""").df()
print(h7_price)
print(h7_basket)

metrics = [
    ("平均単価", h7_price["avg_price"].tolist(), "{:.4f}"),
    ("平均バスケット合計額", h7_basket["avg_basket_total"].tolist(), "{:.4f}"),
    ("平均購入点数(バスケットあたり)", h7_basket["avg_items"].tolist(), "{:.2f}"),
]
channels = ["店舗", "オンライン"]

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, (title, vals, fmt) in zip(axes, metrics):
    bars = ax.bar(channels, vals, color=[BLUE, ORANGE], width=0.5)
    style_ax(ax, hide_y_grid=False)
    ax.xaxis.grid(False)
    ax.set_title(title, fontsize=10, color=SECONDARY)
    top = max(vals)
    ax.set_ylim(0, top * 1.25)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + top * 0.03, fmt.format(v), ha="center", fontsize=9, color=SECONDARY)
fig.suptitle("H7: チャネル別の単価・バスケット規模の比較", x=0.02, ha="left", fontsize=12, color=PRIMARY)
fig.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


---
## H8: 交換行動仮説 — 同一product_code短期再購入はサイズ/カラー交換の可能性がある(採用・部分的)

同一顧客・同一product_codeの購入を日付順に並べ、直前の購入からの間隔(日数)を計算する。同日(1バスケット内のまとめ買い)は交換ではないため区別し、別日再購入の間隔分布を見る。

In [ ]:
gap_summary = con.execute("""
WITH ordered AS (
  SELECT customer_idx, product_code, t_dat,
         LAG(t_dat) OVER (PARTITION BY customer_idx, product_code ORDER BY t_dat) AS prev_dat
  FROM trans
),
gaps AS (
  SELECT DATEDIFF('day', CAST(prev_dat AS DATE), CAST(t_dat AS DATE)) AS gap_days
  FROM ordered WHERE prev_dat IS NOT NULL
)
SELECT
  COUNT(*) AS n_repeat_events,
  SUM(CASE WHEN gap_days = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS pct_same_day,
  median(gap_days) FILTER (WHERE gap_days > 0) AS median_gap_diffday,
  SUM(CASE WHEN gap_days BETWEEN 1 AND 7 THEN 1 ELSE 0 END) * 100.0
    / SUM(CASE WHEN gap_days > 0 THEN 1 ELSE 0 END) AS pct_within_7d,
  SUM(CASE WHEN gap_days BETWEEN 1 AND 14 THEN 1 ELSE 0 END) * 100.0
    / SUM(CASE WHEN gap_days > 0 THEN 1 ELSE 0 END) AS pct_within_14d
FROM gaps
""").df()
print(gap_summary)

gap_hist = con.execute("""
WITH ordered AS (
  SELECT customer_idx, product_code, t_dat,
         LAG(t_dat) OVER (PARTITION BY customer_idx, product_code ORDER BY t_dat) AS prev_dat
  FROM trans
)
SELECT DATEDIFF('day', CAST(prev_dat AS DATE), CAST(t_dat AS DATE)) AS gap_days
FROM ordered
WHERE prev_dat IS NOT NULL AND DATEDIFF('day', CAST(prev_dat AS DATE), CAST(t_dat AS DATE)) BETWEEN 1 AND 60
""").df()
median_gap = gap_summary.iloc[0]["median_gap_diffday"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(gap_hist["gap_days"], bins=60, color=BLUE, alpha=0.9)
ax.axvline(median_gap, color=RED, linewidth=2, linestyle="--")
ax.text(median_gap + 1, ax.get_ylim()[1] * 0.92, f"中央値 = {median_gap:.0f}日", color=RED, fontsize=9, va="top")
style_ax(ax, hide_y_grid=False)
ax.xaxis.grid(False)
ax.set_xlabel("同一product_code再購入(別日)の間隔(日)")
ax.set_ylabel("イベント数")
ax.set_title("H8: 同一product_code再購入の間隔分布(同日除く)", loc="left", fontsize=12, color=PRIMARY)
fig.tight_layout()
plt.show()


---
## H9: コールドスタート比率(採用・設計に直結)

customers.csv全顧客のうち、学習期間の購入回数が0件・1件の顧客の割合を確認する。これが実質的なコールドスタート顧客であり、フォールバック(人気度ベース候補)の優先度に直結する。

In [ ]:
n_customers_total = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{DATA_DIR}/customers.csv')").df().iloc[0, 0]
n_customers_active = con.execute("SELECT COUNT(DISTINCT customer_idx) FROM trans").df().iloc[0, 0]
n_zero = n_customers_total - n_customers_active

buckets = con.execute("""
WITH per_cust AS (SELECT customer_idx, COUNT(*) AS n FROM trans GROUP BY customer_idx)
SELECT
  CASE
    WHEN n = 1 THEN '1回'
    WHEN n BETWEEN 2 AND 4 THEN '2-4回'
    WHEN n BETWEEN 5 AND 9 THEN '5-9回'
    WHEN n BETWEEN 10 AND 29 THEN '10-29回'
    ELSE '30回以上'
  END AS bucket,
  COUNT(*) AS n_cust
FROM per_cust GROUP BY bucket
""").df().set_index("bucket")["n_cust"]

order = ["0回(未購入)", "1回", "2-4回", "5-9回", "10-29回", "30回以上"]
counts = {"0回(未購入)": n_zero, **buckets.to_dict()}
pcts = [counts[o] * 100.0 / n_customers_total for o in order]
print(f"コールドスタート(0〜1回)合計: {(counts['0回(未購入)'] + counts['1回']) * 100.0 / n_customers_total:.1f}%")

# 順序付きカテゴリのため、単色の明度ランプ(sequential, light->dark)を使う
colors = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281", "#0d366b"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(order, pcts, color=colors, width=0.6)
style_ax(ax, hide_y_grid=False)
ax.xaxis.grid(False)
ax.set_ylabel("顧客数に占める割合 (%)")
for b, v in zip(bars, pcts):
    ax.text(b.get_x() + b.get_width() / 2, v + max(pcts) * 0.02, f"{v:.1f}%", ha="center", fontsize=9, color=SECONDARY)
ax.set_ylim(0, max(pcts) * 1.25)
ax.set_title("H9: コールドスタート比率 — 学習期間の購入回数別 顧客分布", loc="left", fontsize=12, color=PRIMARY)
fig.tight_layout()
plt.show()


---
## H10: 商品ライフサイクル仮説 — 死に筋商品の除外要否(採用)

各articleについて、データ終端(2020-09-22)から最終購入日までの経過週数を計算し、「直近N週間売れていない商品」がカタログ全体の何%を占めるかを見る。人気度ベース候補を直近実績で絞るべきかどうかの判断材料。

In [ ]:
n_articles_master = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{DATA_DIR}/articles.csv')").df().iloc[0, 0]
n_articles_sold = con.execute("SELECT COUNT(DISTINCT article_id) FROM trans").df().iloc[0, 0]
print(f"一度も売れなかったarticle: {n_articles_master - n_articles_sold}件 / {n_articles_master}件"
      f" ({(n_articles_master - n_articles_sold) * 100.0 / n_articles_master:.1f}%)")

recency = con.execute("""
SELECT article_id, DATEDIFF('day', MAX(CAST(t_dat AS DATE)), DATE '2020-09-22') AS days_since_last_sale
FROM trans GROUP BY article_id
""").df()

weeks = [1, 2, 4, 8, 12, 26, 52]
pct_dead = [(recency["days_since_last_sale"] > w * 7).mean() * 100 for w in weeks]
for w, p in zip(weeks, pct_dead):
    print(f"直近{w}週間以内に売れていない商品の割合: {p:.1f}%")

x_pos = list(range(len(weeks)))
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x_pos, pct_dead, color=BLUE, linewidth=2, marker="o", markersize=5)
for xp, p in zip(x_pos, pct_dead):
    ax.text(xp, p + 3, f"{p:.0f}%", ha="center", fontsize=9, color=SECONDARY)
style_ax(ax, hide_y_grid=False)
ax.xaxis.grid(False)
ax.set_xlabel("直近何週間、売れていないか(週)")
ax.set_ylabel("該当する商品の割合 (%)")
ax.set_xticks(x_pos)
ax.set_xticklabels([str(w) for w in weeks])
ax.set_xlim(-0.4, len(weeks) - 0.6)
ax.set_ylim(0, 100)
ax.set_title("H10: 商品ライフサイクル — 最終購入からの経過週数別カバレッジ", loc="left", fontsize=12, color=PRIMARY)
fig.tight_layout()
plt.show()


---
## H11: 給料日効果(採用)

day-of-month(1〜31日)ごとに取引件数を集計し、観測日数で正規化した「1日あたり平均取引件数」を、全期間平均=100として指数化する。

In [ ]:
dom = con.execute("""
SELECT DAY(CAST(t_dat AS DATE)) AS dom, COUNT(*) AS n, COUNT(DISTINCT CAST(t_dat AS DATE)) AS n_days_observed
FROM trans GROUP BY dom ORDER BY dom
""").df()
dom["avg_per_day"] = dom["n"] / dom["n_days_observed"]
overall_avg = dom["avg_per_day"].mean()
dom["idx"] = dom["avg_per_day"] / overall_avg * 100
print(dom[["dom", "idx"]])

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(dom["dom"], dom["idx"], color=BLUE, linewidth=2, marker="o", markersize=4)
ax.axhline(100, color=MUTED, linewidth=1.2, linestyle="--")
ax.axvspan(22, 29, color=ORANGE, alpha=0.08)
ax.text(22.3, 128, "給料日帯(22〜29日)", color=ORANGE, fontsize=9, va="top")
style_ax(ax, hide_y_grid=False)
ax.xaxis.grid(False)
ax.set_xlabel("日(day of month)")
ax.set_ylabel("平均取引件数(全期間平均=100)")
ax.set_xlim(0.5, 31.5)
ax.set_xticks(range(1, 32, 2))
ax.set_title("H11: 給料日効果 — 日別(day-of-month)取引件数インデックス", loc="left", fontsize=12, color=PRIMARY)
fig.tight_layout()
plt.show()


---
## まとめと次のステップ

H1〜H5, H7〜H11はすべて`00_md/hypothesis_driven_design.md`(3節・6節)の数値と整合し、採用。H6は引き続き直接検証不可(price列が匿名化スケーリング値のため)。

これで`00_md/hypothesis_driven_design.md`第6節の未検証仮説はすべて解消済み。次は`02_feature_engineering.ipynb`でStage1候補生成ロジック(自己リピート・セグメント別人気度・共起・直近トレンド・死に筋フィルタ・コールドスタート向けフォールバック)とStage2特徴量の実装に進む。